In [1]:
import subprocess
import os
import pandas as pd
import os
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

/Users/jeff/Desktop/6750/spring-2026-a03-XingYx12/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:

def execute_bash_query(command: str):
    """Executes bash commands and returns output."""
    try:
        process = subprocess.run(
            command.strip(), shell=True, capture_output=True, text=True, timeout=15
        )
        if process.returncode == 0:
            return process.stdout if process.stdout else "Success: No output."
        else:
            return f"Bash Error: {process.stderr}"
    except Exception as e:
        return f"System Error: {str(e)}"

def ask_multisource_agent(question: str, max_steps=8):
    history = []
    text_dir = "../data/unstructured/" 
    csv_path = "../data/structured/daily_sales.csv"
    
    # Pre-fetch metadata for LLM
    try:
        df_sample = pd.read_csv(csv_path, nrows=1)
        csv_columns = list(df_sample.columns)
    except:
        csv_columns = "Unknown"

    for i in range(max_steps):
        history_str = "\n".join(history)
        
        planner_prompt = f"""
You are a Senior Data Analyst. Answer: "{question}"

RESOURCES:
- CSV Sales: '{csv_path}' (Columns: {csv_columns})
- Product Text: Directory '{text_dir}' (e.g., SPRT001_product_page.txt)

STRICT OPERATIONAL RULES:
1. PANDAS FILTERING: The 'date' column is a DATETIME object. 
   - Use `df[df['date'].dt.month == 12]` for December. 
   - DO NOT use `.str.contains()` on the date column.
2. DISCOVERY: Always run 'BASH: ls {text_dir}' first to see actual filenames.
3. IDENTIFICATION: Use 'BASH: grep -il "keyword" {text_dir}*.txt' to find the Product ID first.
4. TEXT RETRIEVAL: Use 'BASH: cat {text_dir}<ID>_product_page.txt' to read reviews.
5. CORRELATION: You MUST find the Product ID from Text first, then query sales for that ID in Pandas.

OUTPUT FORMAT:
- Output ONLY the command. NO markdown. No explanation.
- Example: PANDAS: df[df['product_id'] == 'SPRT001']['total_revenue'].sum()
- If you have definitive evidence (numbers + text), output 'FINAL_ANSWER'.

HISTORY:
{history_str if history else "Start exploration."}
"""
        # Get and Clean Action
        raw_response = llm.invoke(planner_prompt).content.strip()
        action = raw_response.replace("```bash", "").replace("```python", "").replace("```", "")
        action = [line.strip() for line in action.split('\n') if line.strip()][0].rstrip('.') 
        
        if "FINAL_ANSWER" in action.upper():
            break
            
        print(f"Step {i+1} | Action: {action}")
        
        # Execution Engine
        if action.startswith("PANDAS:"):
            try:
                df = pd.read_csv(csv_path)
                df['date'] = pd.to_datetime(df['date']) # Ensure proper type
                code = action.replace("PANDAS:", "").strip()
                # Run eval in a safe namespace
                observation = str(eval(code, {"df": df, "pd": pd}))
            except Exception as e:
                observation = f"Pandas Error: {str(e)}. Use simple logic like df[df['date'].dt.month == 12]"
        elif action.startswith("BASH:"):
            cmd = action.replace("BASH:", "").strip()
            observation = execute_bash_query(cmd)
        else:
            observation = f"Format Error: Use 'PANDAS:' or 'BASH:'. Got: {action}"

        if len(observation) > 2000:
            observation = observation[:1000] + "\n...[Truncated]...\n" + observation[-1000:]
            
        history.append(f"Step {i+1} Action: {action}\nObservation: {observation}")

    
    final_prompt = f"""
Write a professional final report for: "{question}"
BASED ONLY ON THE OBSERVATIONS BELOW. 

OBSERVATIONS:
{history_str}

STRICT SYNTHESIS RULES:
1. NEVER make up numbers. If a number is not in the Observations, you DO NOT have it.
2. If Pandas failed, state "Sales data unavailable". Do not guess volume.
3. For Q6: Specifically state which product (ID) you found in Text and what its sales were in the West region from the Pandas step. 
4. If no product matched "fitness" in the Text search, admit it.
"""
    return llm.invoke(final_prompt).content

questions = [
    "What was the total revenue for Electronics category in December 2024?",
    "Which region had the highest sales volume?",
    "What are the key features of the Wireless Bluetooth Headphones?",
    "What do customers say about the Air Fryer's ease of cleaning?",
    "Which product has the best customer reviews and how well is it selling?",
    "I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?"
]

results = []
for q in questions:
    print(f"\nAnalyzing: {q}")
    ans = ask_multisource_agent(q)
    results.append(f"Q: {q}\nA: {ans}")

# Final Save
with open("part2_results.txt", "w", encoding="utf-8") as f:

    divider = "\n\n" + "="*60 + "\n\n"
    f.write(divider.join(results))


Analyzing: What was the total revenue for Electronics category in December 2024?
Step 1 | Action: BASH: ls ../data/unstructured/
Step 2 | Action: BASH: grep -il "Electronics" ../data/unstructured/*.txt
Step 3 | Action: PANDAS: df[df['date'].dt.month == 12][df['category'] == 'Electronics']['total_revenue'].sum()


<string>:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.


Step 4 | Action: PANDAS: df[df['date'].dt.month == 12][df['category'] == 'Electronics']['total_revenue'].sum()


<string>:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.


Step 5 | Action: PANDAS: df[df['date'].dt.month == 12][df['category'] == 'Electronics']['total_revenue'].sum()


...

Step 7 | Action: PANDAS: df[df['date'].dt.month == 12][df['category'] == 'Electronics']['total_revenue'].sum()


<string>:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.


Step 8 | Action: PANDAS: df[df['date'].dt.month == 12][df['category'] == 'Electronics']['total_revenue'].sum()


<string>:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.



Analyzing: Which region had the highest sales volume?
Step 1 | Action: BASH: ls ../data/unstructured/
Step 2 | Action: BASH: grep -il "SPRT001" ../data/unstructured/*.txt
Step 3 | Action: BASH: grep -il "SPRT001" ../data/unstructured/*.txt
Step 4 | Action: PANDAS: df[df['region'] == 'North America']['total_revenue'].sum()
Step 5 | Action: BASH: grep -il "keyword" ../data/unstructured/*.txt
Step 6 | Action: BASH: grep -il "keyword" ../data/unstructured/*.txt
Step 7 | Action: BASH: grep -il "keyword" ../data/unstructured/*.txt
Step 8 | Action: BASH: grep -il "keyword" ../data/unstructured/*.txt

Analyzing: What are the key features of the Wireless Bluetooth Headphones?
Step 1 | Action: BASH: ls ../data/unstructured/
Step 2 | Action: BASH: grep -il "Wireless" ../data/unstructured/*.txt
Step 3 | Action: BASH: grep -il "Bluetooth" ../data/unstructured/ELEC001_product_page.txt
Step 4 | Action: BASH: cat ../data/unstructured/ELEC001_product_page.txt
Step 5 | Action: PANDAS: df[df['product_id